# GeoEngine Dataset Import (system B)

Re-creates the datasets of a package written by `geoengine_dataset_export.ipynb`. The package
must already lie in a GeoEngine **volume** on this instance (a folder listed under
`[data] volumes` in its settings), in the sub-folder `VOLUME_SUBDIR`.

Each dataset folder holds a `dataset.json` with the bodies of `POST /dataset` and
`POST /dataset/{name}/tiles`. The import creates the dataset on the volume and adds its tiles
with the file paths prefixed by the folder's place in the volume, as `ge.import_dataset` does.
On top of that it checks every checksum before writing anything and re-runs the spot checks
afterwards. GeoEngine grants read access to registered and anonymous users. The dataset keeps
its name, so workflows that reference it (such as the BioIS backend) work unchanged.

Creating datasets on a volume requires an admin login.

In [ ]:
import hashlib
import json
from pathlib import Path

import geoengine as ge
import requests

try:
    import tomllib
except ModuleNotFoundError:  # Python < 3.11
    import tomli as tomllib

## Configuration

`IMPORT_DIR` is where this notebook reads the package to verify it; on system B it is
normally the volume folder itself (`<volume path>/<VOLUME_SUBDIR>`).

In [ ]:
GEOENGINE_URL = "http://localhost:3030/api"
GEOENGINE_USER = "admin@localhost"
GEOENGINE_PASSWORD = "adminadmin"

VOLUME_NAME = "dev_data"
VOLUME_SUBDIR = "nexgddp_yearly"
IMPORT_DIR = Path("/home/droenner/dev_data/nexgddp_yearly")

# Existing datasets are skipped; set to True to delete and re-create them.
REPLACE_EXISTING = False
# Appended to every dataset name. Only for a round-trip test on the instance the package came
# from; leave empty on system B so names (and BioIS) stay unchanged.
NAME_SUFFIX = ""

DATASET_FILE = "dataset.json"

# BioIS: every model/scenario in this registry must have all its yearly indicator datasets.
BIOIS_REGISTRY_PATH = Path("../backend/conf/nexgddp_models.toml")
BIOIS_INDICATORS = ["heatDays", "iceDays", "tropicalNights", "frostDays", "dryDays", "heavyRainDays"]

In [ ]:
ge.initialize(GEOENGINE_URL, credentials=(GEOENGINE_USER, GEOENGINE_PASSWORD))
http = requests.Session()
http.headers.update(ge.get_session().auth_header)


def api(method, path, **kwargs):
    # Call the GeoEngine API and return the JSON body; raise with the server's message.
    response = http.request(method, f"{GEOENGINE_URL}/{path}", timeout=600, **kwargs)
    if not response.ok:
        raise RuntimeError(f"{method} {path}: {response.status_code} {response.text}")
    return response.json() if response.content else None


def dataset_or_none(name):
    response = http.get(f"{GEOENGINE_URL}/dataset/{name}", timeout=60)
    return response.json() if response.ok else None


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


print(api("GET", "info"))

## Check before writing

The volume must exist on this instance, and every file and `dataset.json` must match the
checksums in the manifest. Nothing is created if anything is off.

In [ ]:
volumes = {v["name"]: v.get("path") for v in api("GET", "dataset/volumes")}
assert VOLUME_NAME in volumes, f"no volume {VOLUME_NAME!r} on this instance: {sorted(volumes)}"
print(f"volume {VOLUME_NAME} -> {volumes[VOLUME_NAME]}; package expected in {VOLUME_SUBDIR}/")

manifest = json.loads((IMPORT_DIR / "manifest.json").read_text())
print(f"package from {manifest['source']['url']} ({manifest['source']['info']}), exported {manifest['exportedAt']}")
for entry in manifest["datasets"]:
    assert sha256(IMPORT_DIR / entry["name"] / DATASET_FILE) == entry["definitionSha256"], entry["name"]
    for f in entry["files"]:
        assert sha256(IMPORT_DIR / f["path"]) == f["sha256"], f"{f['path']}: checksum mismatch"
print(f"checksums OK for {len(manifest['datasets'])} datasets")

## Import

In [ ]:
report = {"imported": [], "skipped": [], "replaced": []}
for entry in manifest["datasets"]:
    dataset = json.loads((IMPORT_DIR / entry["name"] / DATASET_FILE).read_text())
    name = entry["name"] + NAME_SUFFIX
    definition = dataset["definition"]
    definition["properties"]["name"] = name
    folder_in_volume = f"{VOLUME_SUBDIR}/{entry['name']}"
    tiles = dataset["tiles"]
    for tile in tiles:
        tile["params"]["filePath"] = f"{folder_in_volume}/{tile['params']['filePath']}"

    if dataset_or_none(name) is not None:
        if not REPLACE_EXISTING:
            report["skipped"].append(name)
            continue
        api("DELETE", f"dataset/{name}")
        report["replaced"].append(name)

    api("POST", "dataset", json={"dataPath": {"volume": VOLUME_NAME}, "definition": definition})
    api("POST", f"dataset/{name}/tiles", json=tiles)
    report["imported"].append(name)
    print(f"imported {name}: {len(tiles)} tiles")

print({k: len(v) for k, v in report.items()})

## Verify

The spot checks recorded on system A must give the same values here, read through a
`RasterVectorJoin` at the point, as the BioIS backend does.

In [ ]:
def point_values(dataset, lon, lat, year):
    graph = {"type": "Vector", "operator": {
        "type": "RasterVectorJoin",
        "params": {
            "names": {"type": "names", "values": ["value"]},
            "featureAggregation": "first",
            "temporalAggregation": "none",
        },
        "sources": {
            "vector": {"type": "MockPointSource", "params": {"points": [{"x": lon, "y": lat}]}},
            "rasters": [{"type": "MultiBandGdalSource", "params": {"data": dataset}}],
        },
    }}
    graph_id = api("POST", "processingGraphs", json=graph)["id"]
    d = 0.0001
    collection = api("GET", f"wfs/{graph_id}", params={
        "service": "WFS", "version": "2.0.0", "request": "GetFeature", "typeNames": graph_id,
        "bbox": f"{lon - d},{lat - d},{lon + d},{lat + d}", "srsName": "EPSG:4326",
        "time": f"{year}-01-01T00:00:00Z/{year + 1}-01-01T00:00:00Z",
    })
    return [f["properties"]["value"] for f in collection["features"]]


for entry in manifest["datasets"]:
    assert dataset_or_none(entry["name"] + NAME_SUFFIX) is not None, f"{entry['name']} missing"

mismatches = []
for check in manifest["spotchecks"]:
    values = point_values(check["dataset"] + NAME_SUFFIX, check["lon"], check["lat"], check["year"])
    if values != [check["value"]]:
        mismatches.append({**check, "got": values})
report["spotcheckMismatches"] = mismatches
assert not mismatches, mismatches
print(f"{len(manifest['spotchecks'])} spot checks match system A")

## BioIS model registry

The backend offers each model/scenario of `nexgddp_models.toml` and needs its six yearly
indicator datasets. Every one of them must exist here.

In [ ]:
with open(BIOIS_REGISTRY_PATH, "rb") as fh:
    registry = tomllib.load(fh).get("model", [])
missing = [
    f"nexgddp_cmip6_{m['id']}_{s}_{i}_yearly"
    for m in registry for s in m["scenarios"] for i in BIOIS_INDICATORS
    if dataset_or_none(f"nexgddp_cmip6_{m['id']}_{s}_{i}_yearly") is None
]
report["registryMissing"] = missing
Path("out").mkdir(exist_ok=True)
Path("out/import_report.json").write_text(json.dumps(report, indent=1))
if missing:
    print(f"{len(missing)} datasets the BioIS registry needs are missing, e.g. {missing[:3]}")
else:
    print(f"all {len(registry)} registered models have their yearly datasets")